### Question: Predict type of blood sugar level for given patient

In [1]:
# ---------------------------------------------------------
# MULTI-MODEL BLOOD SUGAR CLASSIFICATION
# ---------------------------------------------------------

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------

df = pd.read_excel("Team6_DataDynamos_Python-Hackathon_MAY2026_V2.xlsx")

# ---------------------------------------------------------
# CREATE FUTURE GLUCOSE
# ---------------------------------------------------------

df['future_glucose'] = df['glucose'].shift(-1)
df = df.dropna()

# ---------------------------------------------------------
# CREATE 4 CLASSES
# ---------------------------------------------------------

def glucose_category(x):
    if x < 70:
        return 0   # Low
    elif x <= 180:
        return 1   # Normal
    elif x <= 250:
        return 2   # High
    else:
        return 3   # Critical

df['glucose_category'] = df['future_glucose'].apply(glucose_category)

# ---------------------------------------------------------
# FEATURES & TARGET
# ---------------------------------------------------------

features = [
    'glucose',
    'basal_rate',
    'bolus_volume_delivered',
    'carb_input',
    'calories',
    'steps',
    'heart_rate',
    'average_sleep_duration_(hrs)'
]

X = df[features]
y = df['glucose_category']

# ---------------------------------------------------------
# TRAIN TEST SPLIT
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# ---------------------------------------------------------
# MODELS TO COMPARE
# ---------------------------------------------------------

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True)
}

# ---------------------------------------------------------
# TRAIN & EVALUATE
# ---------------------------------------------------------

results = []

for name, model in models.items():

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)

    results.append((name, acc, model))

    print("\n------------------------------")
    print("Model:", name)
    print("Accuracy:", round(acc, 4))

# ---------------------------------------------------------
# SELECT BEST MODEL
# ---------------------------------------------------------

best_model_name, best_accuracy, best_model = max(
    results,
    key=lambda x: x[1]
)

print("\n================================================")
print("BEST MODEL SELECTED")
print("================================================")

print("Best Model:", best_model_name)
print("Best Accuracy:", round(best_accuracy, 4))

# ---------------------------------------------------------
# DETAILED REPORT OF BEST MODEL
# ---------------------------------------------------------

y_pred_best = best_model.predict(X_test)

print("\nClassification Report (Best Model):\n")
print(classification_report(y_test, y_pred_best))

# ---------------------------------------------------------
# TEST NEW PATIENT
# ---------------------------------------------------------

new_patient = pd.DataFrame({

    'glucose': [240],
    'basal_rate': [1.2],
    'bolus_volume_delivered': [3],
    'carb_input': [85],
    'calories': [700],
    'steps': [800],
    'heart_rate': [98],
    'average_sleep_duration_(hrs)': [5]

})

prediction = best_model.predict(new_patient)
probability = best_model.predict_proba(new_patient)

label_map = {
    0: "Low",
    1: "Normal",
    2: "High",
    3: "Critical"
}

print("\n================================================")
print("NEW PATIENT PREDICTION (BEST MODEL)")
print("================================================")

print("Predicted Category:", label_map[prediction[0]])
print("\nProbabilities:\n", probability)

C:\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



------------------------------
Model: Logistic Regression
Accuracy: 0.9288

------------------------------
Model: Decision Tree
Accuracy: 0.9572

------------------------------
Model: Random Forest
Accuracy: 0.9702

------------------------------
Model: KNN
Accuracy: 0.9676

------------------------------
Model: SVM
Accuracy: 0.9713

BEST MODEL SELECTED
Best Model: SVM
Best Accuracy: 0.9713

Classification Report (Best Model):

              precision    recall  f1-score   support

           0       0.93      0.91      0.92      4063
           1       0.98      0.99      0.98     44351
           2       0.95      0.94      0.95     10315
           3       0.95      0.95      0.95      3150

    accuracy                           0.97     61879
   macro avg       0.95      0.95      0.95     61879
weighted avg       0.97      0.97      0.97     61879


NEW PATIENT PREDICTION (BEST MODEL)
Predicted Category: Low

Probabilities:
 [[0.24539671 0.36286546 0.11965998 0.27207786]]
